# Phase 4: Model Development

## 1. Project Setup

### Objective
The objective of this phase is to develop and compare traditional machine learning models for predicting household electricity consumption 10 minutes into the future.

### Approach
We will begin with a simple persistence baseline and gradually experiment with regression algorithms such as Linear Regression, Ridge, Lasso, Decision Tree, Random Forest, and Gradient Boosting.

### Evaluation
Models will be evaluated using MAE, RMSE, and MAPE on chronologically separated test data.

In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
data_path = Path(r"D:\college ml project\EnergySense\data\processed\X_features.csv")
df_features = pd.read_csv(data_path)

data_output=Path(r"D:\college ml project\EnergySense\data\processed\y_target.csv")
df_output=pd.read_csv(data_output)
df_features

,timestamp,hour,day,month,weekends,lag_1,lag_10,lag_30,lag_60,lag_5,rolling_mean_5,rolling_mean_10,rolling_mean_30,rolling_std_10,Global_active_power
0,2006-12-16 18:24:00,18.0,5.0,12.0,1.0,2.926,2.388,2.720,4.216,2.852,3.0360,3.5828,3.892867,0.769995,3.452
1,2006-12-16 18:25:00,18.0,5.0,12.0,1.0,3.452,4.598,3.758,5.360,2.928,3.4244,3.6100,3.929933,0.813421,4.870
2,2006-12-16 18:26:00,18.0,5.0,12.0,1.0,4.870,4.524,4.342,5.374,2.940,3.8100,3.6444,3.947467,0.862182,4.868
3,2006-12-16 18:27:00,18.0,5.0,12.0,1.0,4.868,4.202,4.512,5.388,2.934,4.1964,3.7108,3.959267,0.932590,4.866
4,2006-12-16 18:28:00,18.0,5.0,12.0,1.0,4.866,4.472,4.058,3.666,2.926,4.2464,3.5812,3.929867,0.904688,3.176
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2048265,2010-11-26 20:48:00,20.0,4.0,11.0,0.0,0.910,0.976,1.084,1.242,0.988,0.9368,0.9620,1.000400,0.036234,0.912
2048266,2010-11-26 20:49:00,20.0,4.0,11.0,0.0,0.912,0.984,1.086,1.218,0.982,0.9300,0.9584,0.995800,0.035588,0.948
2048267,2010-11-26 20:50:00,20.0,4.0,11.0,0.0,0.948,0.986,1.082,1.234,0.972,0.9752,0.9796,0.999667,0.084031,1.198
2048268,2010-11-26 20:51:00,20.0,4.0,11.0,0.0,1.198,0.990,1.086,1.400,0.908,0.9984,0.9830,0.997600,0.085178,1.024


In [41]:
df_features.shape

(2048270, 15)

In [42]:
df_output


,timestamp,target_10min
0,2006-12-16 18:24:00,3.538
1,2006-12-16 18:25:00,6.072
2,2006-12-16 18:26:00,4.536
3,2006-12-16 18:27:00,4.408
4,2006-12-16 18:28:00,2.912
...,...,...
2048265,2010-11-26 20:48:00,0.946
2048266,2010-11-26 20:49:00,0.944
2048267,2010-11-26 20:50:00,0.938
2048268,2010-11-26 20:51:00,0.934


In [43]:
df_features.index

RangeIndex(start=0, stop=2048270, step=1)

In [44]:
df_output.index

RangeIndex(start=0, stop=2048270, step=1)

In [45]:
df_features['timestamp'] = pd.to_datetime(
        df_features['timestamp'], format='%Y-%m-%d %H:%M:%S', errors='raise'
    )
df_features = df_features.set_index('timestamp')
df_features.index

DatetimeIndex(['2006-12-16 18:24:00', '2006-12-16 18:25:00',
               '2006-12-16 18:26:00', '2006-12-16 18:27:00',
               '2006-12-16 18:28:00', '2006-12-16 18:29:00',
               '2006-12-16 18:30:00', '2006-12-16 18:31:00',
               '2006-12-16 18:32:00', '2006-12-16 18:33:00',
               ...
               '2010-11-26 20:43:00', '2010-11-26 20:44:00',
               '2010-11-26 20:45:00', '2010-11-26 20:46:00',
               '2010-11-26 20:47:00', '2010-11-26 20:48:00',
               '2010-11-26 20:49:00', '2010-11-26 20:50:00',
               '2010-11-26 20:51:00', '2010-11-26 20:52:00'],
              dtype='datetime64[ns]', name='timestamp', length=2048270, freq=None)

In [46]:
df_output['timestamp'] = pd.to_datetime(
        df_output['timestamp'], format='%Y-%m-%d %H:%M:%S', errors='raise'
    )
df_output = df_output.set_index('timestamp')
df_output.index

DatetimeIndex(['2006-12-16 18:24:00', '2006-12-16 18:25:00',
               '2006-12-16 18:26:00', '2006-12-16 18:27:00',
               '2006-12-16 18:28:00', '2006-12-16 18:29:00',
               '2006-12-16 18:30:00', '2006-12-16 18:31:00',
               '2006-12-16 18:32:00', '2006-12-16 18:33:00',
               ...
               '2010-11-26 20:43:00', '2010-11-26 20:44:00',
               '2010-11-26 20:45:00', '2010-11-26 20:46:00',
               '2010-11-26 20:47:00', '2010-11-26 20:48:00',
               '2010-11-26 20:49:00', '2010-11-26 20:50:00',
               '2010-11-26 20:51:00', '2010-11-26 20:52:00'],
              dtype='datetime64[ns]', name='timestamp', length=2048270, freq=None)

In [47]:
df_features.index.equals(df_output.index)

True

In [48]:
df_features.shape

(2048270, 14)

In [49]:
df_features

,hour,day,month,weekends,lag_1,lag_10,lag_30,lag_60,lag_5,rolling_mean_5,rolling_mean_10,rolling_mean_30,rolling_std_10,Global_active_power
timestamp,,,,,,,,,,,,,,
2006-12-16 18:24:00,18.0,5.0,12.0,1.0,2.926,2.388,2.720,4.216,2.852,3.0360,3.5828,3.892867,0.769995,3.452
2006-12-16 18:25:00,18.0,5.0,12.0,1.0,3.452,4.598,3.758,5.360,2.928,3.4244,3.6100,3.929933,0.813421,4.870
2006-12-16 18:26:00,18.0,5.0,12.0,1.0,4.870,4.524,4.342,5.374,2.940,3.8100,3.6444,3.947467,0.862182,4.868
2006-12-16 18:27:00,18.0,5.0,12.0,1.0,4.868,4.202,4.512,5.388,2.934,4.1964,3.7108,3.959267,0.932590,4.866
2006-12-16 18:28:00,18.0,5.0,12.0,1.0,4.866,4.472,4.058,3.666,2.926,4.2464,3.5812,3.929867,0.904688,3.176
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2010-11-26 20:48:00,20.0,4.0,11.0,0.0,0.910,0.976,1.084,1.242,0.988,0.9368,0.9620,1.000400,0.036234,0.912
2010-11-26 20:49:00,20.0,4.0,11.0,0.0,0.912,0.984,1.086,1.218,0.982,0.9300,0.9584,0.995800,0.035588,0.948
2010-11-26 20:50:00,20.0,4.0,11.0,0.0,0.948,0.986,1.082,1.234,0.972,0.9752,0.9796,0.999667,0.084031,1.198


In [50]:
X=df_features

In [51]:
X.shape

(2048270, 14)

In [52]:
y=df_output['target_10min']

In [53]:
y

timestamp
2006-12-16 18:24:00    3.538
2006-12-16 18:25:00    6.072
2006-12-16 18:26:00    4.536
2006-12-16 18:27:00    4.408
2006-12-16 18:28:00    2.912
                       ...  
2010-11-26 20:48:00    0.946
2010-11-26 20:49:00    0.944
2010-11-26 20:50:00    0.938
2010-11-26 20:51:00    0.934
2010-11-26 20:52:00    0.932
Name: target_10min, Length: 2048270, dtype: float64

In [54]:
X.shape

(2048270, 14)

In [55]:
y.shape

(2048270,)

In [56]:
X.index.equals(y.index)

True

In [57]:
X.columns

Index(['hour', 'day', 'month', 'weekends', 'lag_1', 'lag_10', 'lag_30',
       'lag_60', 'lag_5', 'rolling_mean_5', 'rolling_mean_10',
       'rolling_mean_30', 'rolling_std_10', 'Global_active_power'],
      dtype='object')

In [60]:
split_idx=int(len(X)*0.70)
split_timestamp=X.index[split_idx]
validation_idx=int(len(X)*0.85)
validation_boundary=X.index[validation_idx]
validation_boundary

Timestamp('2010-04-18 18:09:00')

In [61]:
# Training: Target must be before the 70% boundary
train_mask = (
    X.index + pd.Timedelta(minutes=10)
) < split_timestamp


# Validation: Prediction starts at 70%, target stays before 85%
val_mask = (
    (X.index >= split_timestamp) &
    (X.index + pd.Timedelta(minutes=10) < validation_boundary)
)


# Testing: Prediction starts at the 85% boundary
test_mask = X.index >= validation_boundary
print("Training samples:", train_mask.sum())
print("Validation samples:", val_mask.sum())
print("Testing samples:", test_mask.sum())

print("Total samples:", train_mask.sum() + val_mask.sum() + test_mask.sum())
print("Original samples:", len(X))

Training samples: 1433779
Validation samples: 307230
Testing samples: 307241
Total samples: 2048250
Original samples: 2048270


In [62]:
# Training data
X_train = X.loc[train_mask]
y_train = y.loc[train_mask]

# Validation data
X_val = X.loc[val_mask]
y_val = y.loc[val_mask]

# Testing data
X_test = X.loc[test_mask]
y_test = y.loc[test_mask]

In [63]:
print(
    "Train target ends before validation:",
    (X_train.index + pd.Timedelta(minutes=10)).max() < split_timestamp
)

print(
    "Validation target ends before testing:",
    (X_val.index + pd.Timedelta(minutes=10)).max() < validation_boundary
)

print(
    "All indexes aligned:",
    X_train.index.equals(y_train.index),
    X_val.index.equals(y_val.index),
    X_test.index.equals(y_test.index)
)

Train target ends before validation: True
Validation target ends before testing: True
All indexes aligned: True True True


In [75]:
baseline_val_predictions=X_val['Global_active_power']
baseline_val_predictions

timestamp
2009-09-13 17:13:00    1.424
2009-09-13 17:14:00    1.442
2009-09-13 17:15:00    1.438
2009-09-13 17:16:00    1.422
2009-09-13 17:17:00    1.434
                       ...  
2010-04-18 17:54:00    0.492
2010-04-18 17:55:00    0.514
2010-04-18 17:56:00    0.512
2010-04-18 17:57:00    0.512
2010-04-18 17:58:00    0.512
Name: Global_active_power, Length: 307230, dtype: float64

In [72]:
baseline_val_predictions.head(10)

timestamp
2009-09-13 17:13:00    1.424
2009-09-13 17:14:00    1.442
2009-09-13 17:15:00    1.438
2009-09-13 17:16:00    1.422
2009-09-13 17:17:00    1.434
2009-09-13 17:18:00    1.426
2009-09-13 17:19:00    1.396
2009-09-13 17:20:00    1.412
2009-09-13 17:21:00    1.356
2009-09-13 17:22:00    1.338
Name: Global_active_power, dtype: float64

In [74]:
y_val.head(10)

timestamp
2009-09-13 17:13:00    1.424
2009-09-13 17:14:00    1.402
2009-09-13 17:15:00    1.396
2009-09-13 17:16:00    1.392
2009-09-13 17:17:00    1.388
2009-09-13 17:18:00    1.388
2009-09-13 17:19:00    1.388
2009-09-13 17:20:00    1.382
2009-09-13 17:21:00    1.326
2009-09-13 17:22:00    1.296
Name: target_10min, dtype: float64

In [90]:
from sklearn.metrics import mean_absolute_error,root_mean_squared_error,mean_squared_error,mean_absolute_percentage_error
baseline_mae=mean_absolute_error(y_val,baseline_val_predictions)
print("MAE:",baseline_mae)
baseline_rmse=root_mean_squared_error(y_val,baseline_val_predictions)
print("RMSE:",baseline_rmse)
baseline_mse=mean_squared_error(y_val,baseline_val_predictions)
print("MSE:",baseline_mse)
baseline_mape=mean_absolute_percentage_error(y_val,baseline_val_predictions)
print("MAPE:",baseline_mape)

MAE: 0.3499187058555479
RMSE: 0.6908274190541519
MSE: 0.4772425229170208
MAPE: 0.3347142371669093


In [92]:
from sklearn.linear_model import LinearRegression
lr_model=LinearRegression()
lr_model.fit(X_train,y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [97]:
lr_model.coef_

array([ 4.48460202e-03, -6.73413285e-05, -1.43007939e-03,  3.80607037e-02,
       -5.59543303e-02,  6.88363916e-02,  5.74453345e-03,  4.48405137e-02,
       -6.59464142e-02,  1.37871265e-01,  1.26780192e-01,  1.29558326e-01,
       -2.47184333e-01,  5.04262371e-01])

In [99]:
lr_model.intercept_

np.float64(0.10325818992543023)

In [100]:
lr_val_predictions=lr_model.predict(X_val)
lr_val_predictions

array([1.49674675, 1.47386818, 1.46651246, ..., 0.63373267, 0.64164353,
       0.64458186], shape=(307230,))

In [101]:
y_val

timestamp
2009-09-13 17:13:00    1.424
2009-09-13 17:14:00    1.402
2009-09-13 17:15:00    1.396
2009-09-13 17:16:00    1.392
2009-09-13 17:17:00    1.388
                       ...  
2010-04-18 17:54:00    1.004
2010-04-18 17:55:00    1.030
2010-04-18 17:56:00    1.044
2010-04-18 17:57:00    0.480
2010-04-18 17:58:00    0.312
Name: target_10min, Length: 307230, dtype: float64

In [102]:
from sklearn.metrics import mean_absolute_error,root_mean_squared_error,mean_squared_error,mean_absolute_percentage_error
lr_mae=mean_absolute_error(y_val,lr_val_predictions)
print("MAE:",lr_mae)
lr_rmse=root_mean_squared_error(y_val,lr_val_predictions)
print("RMSE:",lr_rmse)
lr_mse=mean_squared_error(y_val,lr_val_predictions)
print("MSE:",lr_mse)
lr_mape=mean_absolute_percentage_error(y_val,lr_val_predictions)
print("MAPE:",lr_mape)

MAE: 0.3523612376844683
RMSE: 0.6355035754826579
MSE: 0.40386479445124224
MAPE: 0.39348733391741103


In [103]:
from sklearn.linear_model import Ridge
ridge_model=Ridge(alpha=1.0)
ridge_model.fit(X_train,y_train)

,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [104]:
ridge_val_predictions=ridge_model.predict(X_val)
ridge_val_predictions

array([1.49674834, 1.47387228, 1.46651601, ..., 0.63373244, 0.64164326,
       0.64458156], shape=(307230,))

In [112]:
runscaled_mae=mean_absolute_error(y_val,ridge_val_predictions)
print("MAE:",runscaled_mae)
runscaled_rmse=root_mean_squared_error(y_val,ridge_val_predictions)
print("RMSE:",runscaled_rmse)
runscaled_mse=mean_squared_error(y_val,ridge_val_predictions)
print("MSE:",runscaled_mse)
runscaled_mape=mean_absolute_percentage_error(y_val,ridge_val_predictions)
print("MAPE:",runscaled_mape)

MAE: 0.3523613178905212
RMSE: 0.6355036014066813
MSE: 0.40386482740086205
MAPE: 0.3934875159437545


In [107]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
ridge_pipeline=Pipeline([('scaler', StandardScaler()), ('ridge',Ridge(alpha=1.0))])

In [108]:
ridge_pipeline.fit(X_train, y_train)

,steps,"[('scaler', ...), ('ridge', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None


In [110]:
ridge_pipeline_val_predictions=ridge_pipeline.predict(X_val)

In [113]:
r_mae=mean_absolute_error(y_val,ridge_pipeline_val_predictions)
print("MAE:",r_mae)
r_rmse=root_mean_squared_error(y_val,ridge_pipeline_val_predictions)
print("RMSE:",r_rmse)
r_mse=mean_squared_error(y_val,ridge_pipeline_val_predictions)
print("MSE:",r_mse)
r_mape=mean_absolute_percentage_error(y_val,ridge_pipeline_val_predictions)
print("MAPE:",r_mape)

MAE: 0.35236131129145354
RMSE: 0.6355036120264637
MSE: 0.4038648408986821
MAPE: 0.3934875109611338


In [118]:
from sklearn import linear_model
lasso_pipeline=Pipeline([('scaler', StandardScaler()), ('lasso',linear_model.Lasso(alpha=0.1))])
lasso_pipeline.fit(X_train, y_train)

,steps,"[('scaler', ...), ('lasso', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,alpha,0.1
,fit_intercept,True
,precompute,False
,copy_X,True


In [123]:
lasso_pipeline_val_predictions=lasso_pipeline.predict(X_val)
lasso_mae=mean_absolute_error(y_val,lasso_pipeline_val_predictions)
print("MAE:",lasso_mae)
lasso_rmse=root_mean_squared_error(y_val,lasso_pipeline_val_predictions)
print("RMSE:",lasso_rmse)
lasso_mse=mean_squared_error(y_val,lasso_pipeline_val_predictions)
print("MSE:",lasso_mse)
lasso_mape=mean_absolute_percentage_error(y_val,lasso_pipeline_val_predictions)
print("MAPE:",lasso_mape)

MAE: 0.3854517816971919
RMSE: 0.6484208275176969
MSE: 0.4204495695587349
MAPE: 0.48606075977630514


In [132]:
ans=lasso_pipeline.named_steps['lasso'].coef_==0
ans.sum()

np.int64(10)

In [134]:
from sklearn.tree import DecisionTreeRegressor
dt_model=DecisionTreeRegressor(max_depth=10,min_samples_split=20,min_samples_leaf=10,random_state=42)
dt_model.fit(X_train,y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,10
,min_samples_split,20
,min_samples_leaf,10
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [138]:
dt_model_predictions=dt_model.predict(X_val)
dt_model_predictions

array([1.65289052, 1.65289052, 1.65289052, ..., 0.68781717, 0.68781717,
       0.68781717], shape=(307230,))

In [140]:
dt_model_mae=mean_absolute_error(y_val,dt_model_predictions)
print("MAE:",dt_model_mae)
dt_model_rmse=root_mean_squared_error(y_val,dt_model_predictions)
print("RMSE:",dt_model_rmse)
dt_model_mse=mean_squared_error(y_val,dt_model_predictions)
print("MSE:",dt_model_mse)
dt_model_mape=mean_absolute_percentage_error(y_val,dt_model_predictions)
print("MAPE:",dt_model_mape)

MAE: 0.339128624794331
RMSE: 0.6141963507843293
MSE: 0.377237157316787
MAPE: 0.357863806920326


In [145]:
dt_model_predictions_X=dt_model.predict(X_train)
dt_model_mae_X=mean_absolute_error(y_train,dt_model_predictions_X)
print("MAE:",dt_model_mae_X)

MAE: 0.3293151128525928


In [150]:
from sklearn.ensemble import RandomForestRegressor
rf_model=RandomForestRegressor(n_estimators=100,max_depth=15,min_samples_split=20,min_samples_leaf=10,random_state=42,n_jobs=-1)

In [151]:
rf_model.fit(X_train,y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,15
,min_samples_split,20
,min_samples_leaf,10
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [153]:
rf_model_predictions=rf_model.predict(X_val)

In [154]:
rf_model_mae=mean_absolute_error(y_val,rf_model_predictions)
print("MAE:",rf_model_mae)
rf_model_rmse=root_mean_squared_error(y_val,rf_model_predictions)
print("RMSE:",rf_model_rmse)
rf_model_mse=mean_squared_error(y_val,rf_model_predictions)
print("MSE:",rf_model_mse)
rf_model_mape=mean_absolute_percentage_error(y_val,rf_model_predictions)
print("MAPE:",rf_model_mape)

MAE: 0.33119562523710405
RMSE: 0.5992764430648343
MSE: 0.35913225521243963
MAPE: 0.3516185296922705


In [155]:
rf_model_predictions_X=rf_model.predict(X_train)
rf_model_mae_X=mean_absolute_error(y_train,rf_model_predictions_X)
print("MAE:",rf_model_mae_X)

MAE: 0.2914140861944735


In [160]:
from sklearn.ensemble import GradientBoostingRegressor
gb_model=GradientBoostingRegressor(n_estimators=30,learning_rate=0.05,max_depth=2,min_samples_split=20,
loss='squared_error',
random_state=42)
gb_model.fit(X_train,y_train)

,loss,'squared_error'
,learning_rate,0.05
,n_estimators,30
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,20
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,2
,min_impurity_decrease,0.0
,init,None


In [163]:
gb_model_predictions=gb_model.predict(X_val)

In [165]:
gb_model_mae=mean_absolute_error(y_val,gb_model_predictions)
print("MAE:",gb_model_mae)
gb_model_rmse=root_mean_squared_error(y_val,gb_model_predictions)
print("RMSE:",gb_model_rmse)
gb_model_mse=mean_squared_error(y_val,gb_model_predictions)
print("MSE:",gb_model_mse)
gb_model_mape=mean_absolute_percentage_error(y_val,gb_model_predictions)
print("MAPE:",gb_model_mape)

MAE: 0.39060031716392923
RMSE: 0.6625996861736909
MSE: 0.4390383441174737
MAPE: 0.5061866222921876
